In [1]:
import cudaq

/tmp/ipykernel_471/980359867.py:1: FutureWarning: The CUDA-Q `sample` and `observe` algorithmic primitives will change in a future release. Existing code may require updates. See https://nvidia.github.io/cuda-quantum/latest/using/migration/upcoming_changes.html for details.
  import cudaq


We can use different backends: either actual QPU or simulated QPU. Then simulated QPU can be in differnt types. 

In [4]:
# Explicitly use the CPU simulator.
cudaq.set_target("qpp-cpu")
print("Running on:", cudaq.get_target().name)

Running on: qpp-cpu


Lets create a quantum circuit as a cudaq kernel (@cudaq.kernel). A cudaq kernel would be run on the selected cudaq target.

We will first create bell state generation circuit which will generate quntum entangled state.

In [9]:
@cudaq.kernel
def bell():

    q = cudaq.qvector(2)

     # Put q0 into superposition by applying hadamard gate to first qubit
    h(q[0])

    # Apply CNOT gate with q0 as control and q1 as the target.
    x.ctrl(q[0], q[1])

    # Measure both qubits.
    mz(q)

Now we can run this cudaq kernel calculating sampling across shots.

In [11]:
result = cudaq.sample(
    bell,
    shots_count=1000
)

print(result)

{ 00:524 11:476 }



Now lets inspect the quantum state instead of sampling it:

In [12]:
@cudaq.kernel
def bell():
    q = cudaq.qvector(2)

    h(q[0])
    x.ctrl(q[0], q[1])

In [13]:
state = cudaq.get_state(bell)

print(state)

(0.707107,0)
       (0,0)
       (0,0)
(0.707107,0)



Now lets use observe() which is mor relavent to hybrid algorithms.

zz = P(00)+P(11)-P(01)-P(10).


In [15]:
from cudaq import spin

observable = spin.z(0) * spin.z(1)

result = cudaq.observe(
    bell,
    observable
)

print("<ZZ> =", result.expectation())

<ZZ> = 0.9999999999999998


Now lets try desnity matrix simulation 

In [28]:
cudaq.set_target("density-matrix-cpu")

@cudaq.kernel
def circuit():
    q = cudaq.qvector(2)

    h(q[0])
    x.ctrl(q[0], q[1])

    mz(q)


We will define a simple dpolarizing noise model that will affect H gate (5% depolarization after H gates)

In [29]:
noise = cudaq.NoiseModel()

channel = cudaq.DepolarizationChannel(0.05)

noise.add_all_qubit_channel(
    "h",
    channel
)


In [30]:
result = cudaq.sample(
    circuit,
    shots_count=1000,
    noise_model=noise
)

print(result)

{ 00:499 11:501 }



Now we have two noise channels: one for H and one for CNOT

In [31]:
noise = cudaq.NoiseModel()

# 1Q error after H
noise.add_all_qubit_channel(
    "h",
    cudaq.DepolarizationChannel(0.05)
)

# 2Q depolarization after controlled-X / CNOT
noise.add_all_qubit_channel(
    "x",
    cudaq.Depolarization2(0.10),
    num_controls=1
)

result = cudaq.sample(
    bell,
    shots_count=10000,
    noise_model=noise
)

print(result)

{ 00:4743 01:279 10:274 11:4704 }

